# Lab 02: A Simple Rational Agent

This notebook builds a robot vacuum in a two-room environment. You will compare a simple reflex agent with a model-based agent.

## Helpful links
- [Download VS Code](https://code.visualstudio.com/download)
- [Download Python](https://www.python.org/downloads/)
- [Install the Python Extension Pack](https://marketplace.visualstudio.com/items?itemName=donjayamanne.python-extension-pack)

## Terminal setup
Run this command in the VS Code terminal. Lab 02 uses only Python's built-in `random` module.

```bash
python3 --version
```

## 1. Build the environment
The environment stores the cleanliness of rooms A and B, the agent's location, and a performance score.

In [ ]:
import random

class VacuumEnvironment:
    def __init__(self):
        self.status = {
            'A': random.choice(['Clean', 'Dirty']),
            'B': random.choice(['Clean', 'Dirty'])
        }
        self.agent_location = random.choice(['A', 'B'])
        self.performance = 0

    def percept(self):
        return (self.agent_location, self.status[self.agent_location])

    def execute(self, action):
        if action == 'Suck':
            if self.status[self.agent_location] == 'Dirty':
                self.status[self.agent_location] = 'Clean'
        elif action == 'Right':
            self.agent_location = 'B'
            self.performance -= 1
        elif action == 'Left':
            self.agent_location = 'A'
            self.performance -= 1

        for room in self.status:
            if self.status[room] == 'Clean':
                self.performance += 1

environment = VacuumEnvironment()
print('Initial percept:', environment.percept())
print('Room status:', environment.status)

## 2. Write a simple reflex agent
This agent uses only its current percept. If the room is dirty it sucks; otherwise it moves to the other room.

In [ ]:
def simple_reflex_agent(percept):
    location, status = percept

    if status == 'Dirty':
        return 'Suck'
    if location == 'A':
        return 'Right'
    return 'Left'

print('Action for the current percept:', simple_reflex_agent(environment.percept()))

## 3. Write a model-based agent
This agent remembers what it has observed. When it knows both rooms are clean, it returns `NoOp` instead of moving unnecessarily.

In [ ]:
class ModelBasedAgent:
    def __init__(self):
        self.model = {'A': None, 'B': None}

    def act(self, percept):
        location, status = percept
        self.model[location] = status

        if status == 'Dirty':
            return 'Suck'
        if self.model['A'] == 'Clean' and self.model['B'] == 'Clean':
            return 'NoOp'
        return 'Right' if location == 'A' else 'Left'

model_agent = ModelBasedAgent()
print('Action for the current percept:', model_agent.act(environment.percept()))
print('Agent memory:', model_agent.model)

## 4. Run a fair comparison
The random seed makes both agents face the same starting world. Each step follows the sense-think-act cycle.

In [ ]:
def run_trial(agent_type, steps=6):
    environment = VacuumEnvironment()
    model_agent = ModelBasedAgent() if agent_type == 'model' else None

    for _ in range(steps):
        percept = environment.percept()
        if agent_type == 'reflex':
            action = simple_reflex_agent(percept)
        else:
            action = model_agent.act(percept)
        environment.execute(action)

    return environment.performance

random.seed(42)
reflex_score = run_trial('reflex')

random.seed(42)
model_score = run_trial('model')

print('Simple Reflex Agent performance:', reflex_score)
print('Model-Based Agent performance: ', model_score)

## What to observe
The model-based agent can avoid unnecessary movement after it knows both rooms are clean. The exact scores depend on the environment and number of steps, but the seeded comparison is repeatable.

## Try it yourself
1. Increase `steps` from 6 to 20.
2. Add a third room called C.
3. Make dirt reappear randomly after each action.
4. Add a goal-based agent that plans a route to clean both rooms.